# Physics-Informed Neural Networks for Electromagnetic Field Prediction

## Introduction

While traditional deep learning models learn mappings purely from data, **Physics-Informed Neural Networks (PINNs)** integrate physical laws directly into the learning process {cite}`raissi2019physics,karniadakis2021physics`. This approach is particularly valuable for electromagnetic field prediction where Maxwell's equations provide rigorous governing constraints {cite}`sadiku2014elements,ida2015numerical`.

**Chapter context**: Sections 4-5 developed data-driven CNNs with uncertainty quantification, achieving 0.3-0.6% NMSE on magnetic field prediction. However, these models required 30,000+ FEA training samples and provided no guarantee of physics consistency (e.g., ∇·**B** = 0 may be violated locally).

**PINN alternative**: This section explores an orthogonal paradigm—incorporating Maxwell's equations as soft constraints during training—enabling physics-consistent predictions from sparse data {cite}`raissi2019physics,cai2021physics`.

### Key Advantages of PINNs for Electromagnetics

1. **Physics consistency** {cite}`raissi2019physics,karniadakis2021physics`: Predictions automatically satisfy Maxwell's equations
   - ∇×**H** = **J** (Ampere's law)
   - ∇·**B** = 0 (Gauss's law for magnetism)
   - Boundary conditions enforced via loss function

2. **Data efficiency** {cite}`cai2021physics,yang2021adversarial`: Learn from sparse observations (10-100 points vs. 30,000 for pure data-driven)
   - Physics constraints regularize solution space
   - Prior knowledge reduces dependence on data

3. **Improved extrapolation** {cite}`raissi2019physics`: Better generalization to unseen parameter regimes
   - Physical laws hold beyond training distribution
   - Reduces epistemic uncertainty from Chapter 5

4. **Inverse problem capability** {cite}`raissi2017physics,karniadakis2021physics`: Simultaneously solve for unknowns:
   - Forward: Geometry → field (standard prediction)
   - Inverse: Field measurements → material properties, source currents

### PINN Mathematical Framework

**Traditional supervised learning**:
$$\mathcal{L}_{\text{data}}(\theta) = \frac{1}{N}\sum_{i=1}^{N} ||y_i - f_{\theta}(x_i)||^2$$

**Physics-Informed learning** {cite}`raissi2019physics`:
$$\mathcal{L}_{\text{total}}(\theta) = \underbrace{\mathcal{L}_{\text{data}}(\theta)}_{\text{Match observations}} + \lambda \underbrace{\mathcal{L}_{\text{physics}}(\theta)}_{\text{Satisfy PDEs}}$$

where:
- $\mathcal{L}_{\text{data}}$ = Standard MSE on labeled data (sparse)
- $\mathcal{L}_{\text{physics}}$ = PDE residual penalty at collocation points (dense)
- $\lambda$ = Physics loss weight (hyperparameter, typically 0.1-10)
- $\theta$ = Neural network parameters

:::{important}
**Automatic Differentiation for PDE Enforcement**  

Modern deep learning frameworks (PyTorch, TensorFlow) enable efficient computation of spatial derivatives via automatic differentiation {cite}`baydin2018automatic`:

$$\frac{\partial f_{\theta}(x,y)}{\partial x} = \text{autograd}(f_{\theta}, x)$$

This allows direct evaluation of PDE residuals:
$$\mathcal{R}_{\text{PDE}} = \nabla \times f_{\theta}(x,y) - J(x,y)$$

without manual derivative implementation—a critical enabler for PINNs {cite}`raissi2019physics,karniadakis2021physics`.
:::

## Literature Survey: PINNs in Computational Electromagnetics

### Foundations of Physics-Informed Neural Networks

The concept of Physics-Informed Neural Networks was formalized by Raissi, Perdikaris, and Karniadakis (2019) {cite}`raissi2019physics` in their seminal work demonstrating that neural networks can solve forward and inverse problems governed by partial differential equations. Subsequent developments {cite}`karniadakis2021physics,cai2021physics` established PINNs as a powerful framework for scientific machine learning.

**Core innovations** {cite}`raissi2019physics,baydin2018automatic`:
- **Automatic differentiation**: Compute spatial/temporal derivatives via backpropagation
- **Collocation methods**: Enforce PDEs at randomly sampled points in the domain
- **Soft constraints**: Physics incorporated via loss function penalties (not hard constraints)
- **Multi-task learning**: Simultaneously optimize data fit and physics satisfaction

**Theoretical justification**: PINNs approximate the solution to PDEs by minimizing the L² norm of the PDE residual, equivalent to variational methods in numerical analysis {cite}`karniadakis2021physics,raissi2017physics`.

### Applications in Computational Electromagnetics

#### 1. Magnetostatic Field Prediction

**Barmada et al. (2020)** {cite}`barmada2020deep`: First application of deep learning to transformer magnetic fields
- Problem: 2D magnetostatic field distribution in power transformers
- Approach: Feedforward NN with physics-informed training
- Results: 10,000× speedup vs. FEA with < 1% error
- **Key contribution**: Demonstrated PINNs work for nonlinear B-H characteristics

**Chen et al. (2022)** {cite}`chen2022pinn`: Extended PINNs to rotating electrical machines
- Problem: Interior Permanent Magnet (IPM) motor with saturation
- Innovation: Adaptive collocation sampling near material boundaries
- Results: Accurate torque prediction with only 50 training samples

#### 2. Electrostatic Problems

**Henning & Stewart (2020)** {cite}`henning2020physics`: Solved 2D Poisson's equation for electrostatic potential
- PDE: $\nabla^2 \phi = -\rho/\epsilon_0$
- Achievement: Complex geometries without explicit mesh generation
- Limitation: Struggled with sharp corner singularities

**Stein et al. (2021)** {cite}`stein2021deep`: Applied PINNs to capacitance extraction
- Industrial application: VLSI interconnect parasitic extraction
- Data efficiency: 90% reduction in required field simulations

#### 3. Time-Harmonic Electromagnetic Waves

**Gao et al. (2021)** {cite}`gao2021physics`: Time-harmonic Maxwell's equations for eddy currents
- PDEs: $\nabla \times (\nabla \times \mathbf{E}) - k^2 \mathbf{E} = -j\omega\mu \mathbf{J}$
- Application: Skin effect and proximity effect in conductors
- Frequency range: 50 Hz - 10 kHz (power electronics)
- **Challenge**: Complex-valued networks for phasor representation

**Rasht-Behesht et al. (2022)** {cite}`rasht2022physics`: Full 3D Maxwell solver using PINNs
- Achievement: No meshing required for arbitrary geometries
- Computational cost: 10× longer training than FEA, but reusable for parameter sweeps

#### 4. Inverse Problems in Electromagnetics

**Wang et al. (2022)** {cite}`wang2022inverse`: Material property identification from field measurements
- Problem: Given sparse B-field measurements, infer permeability $\mu_r(x,y)$
- Method: Treat $\mu_r$ as learnable parameters alongside network weights
- Application: Non-destructive testing, defect detection in steel
- Results: Reconstructed $\mu_r$ with 5% accuracy from 20 measurement points

**Karniadakis et al. (2021)** {cite}`karniadakis2021physics`: Comprehensive review of PINN inverse methods
- Source localization: Identify current distribution from external field measurements
- Boundary condition inference: Determine unknown BCs from interior field data

### Comparison with Traditional Numerical Methods

| Aspect | FEA {cite}`ida2015numerical` | Boundary Element Method {cite}`brebbia1984boundary` | Traditional DL | PINNs {cite}`raissi2019physics` |
|--------|-----|-----|----------------|-------|
| **Mesh Required** | Yes (critical) | Boundary only | No | No |
| **Data Requirements** | None (physics solver) | None | High (10k+) | Low (10-100) |
| **Physics Consistency** | Exact (to discretization) | Exact | No guarantee | Enforced (soft) |
| **Inference Speed** | Slow (minutes) | Medium | Very Fast (ms) | Very Fast (ms) |
| **Extrapolation** | N/A | N/A | Poor | Good |
| **Training Cost** | N/A | N/A | Moderate | High (autodiff) |
| **Nonlinear Materials** | Yes | Limited | Yes | Yes |

:::{note}
**When PINNs Excel vs. FEA**  

**PINN advantages** {cite}`cai2021physics,karniadakis2021physics`:
1. **Parametric studies**: Once trained, instant evaluation across parameter space
2. **Inverse problems**: Naturally handle parameter identification
3. **Sparse data**: Augment limited measurements with physics
4. **Mesh-free**: Complex geometries without mesh generation expertise

**FEA advantages** {cite}`ida2015numerical,meeker2015femm`:
1. **Accuracy**: Sub-0.1% error with sufficient mesh refinement
2. **Maturity**: 50+ years of development, extensive validation
3. **Industrial acceptance**: Regulatory approval, standardized workflows
4. **Complex physics**: Multi-physics coupling (thermal, mechanical, electromagnetic)

**Recommendation**: Use PINNs for exploration, FEA for validation.
:::

### Current Limitations and Active Research Directions

**Training difficulties** {cite}`wang2021understanding,wang2021eigenvector`:
1. **Loss balancing**: Choosing $\lambda$ to balance data vs. physics losses
   - Too high: Underfit data (physics dominates)
   - Too low: Violate physics (data dominates)
   - **Solution**: Adaptive weighting schemes {cite}`wang2021understanding`

2. **Multi-scale phenomena** {cite}`jagtap2020extended`:
   - Sharp gradients at boundaries challenge uniform collocation
   - **Solution**: Adaptive sampling, domain decomposition (XPINNs)

3. **Convergence speed**:
   - Standard Adam optimizer often struggles
   - **Solution**: L-BFGS second-order optimization {cite}`liu1989limited`

**Complex geometries** {cite}`sukumar2022exact`:
- Material interfaces with discontinuous derivatives
- **Approaches**: Exact imposition of BCs {cite}`sukumar2022exact`, interface-aware sampling

**Recent methodological advances** {cite}`karniadakis2021physics,cai2021physics`:
- **Extended PINNs (XPINNs)** {cite}`jagtap2020extended`: Domain decomposition for parallel training
- **Conservative PINNs (cPINNs)** {cite}`jagtap2020conservative`: Enforce conservation laws explicitly
- **Multi-fidelity PINNs** {cite}`meng2020multi`: Combine low-fidelity analytical + high-fidelity FEA data
- **Adaptive collocation** {cite}`wu2023comprehensive`: Dynamic sampling based on residual magnitude

:::{tip}
**Emerging Hybrid Approaches**  

Most promising direction: **Combine PINNs with traditional methods** {cite}`yang2021adversarial,geneva2020modeling`:
1. **PINN pretraining + data finetuning**: Physics initialization, then refine with FEA
2. **Multi-fidelity**: PINN for cheap approximation, FEA for accuracy verification
3. **Physics-guided active learning**: Use PINN residuals to guide FEA sample collection

**Result**: Physics consistency + data accuracy, achieving best of both paradigms.
:::

## Comparison: Traditional Deep Learning vs Physics-Informed Approach

### Traditional Deep Learning (Sections 4-5: Pure Data-Driven CNN)

In Sections 4-5, we developed a **pure data-driven CNN** for magnetic field prediction, representative of standard deep learning practice {cite}`lecun2015deep,goodfellow2016deep`:

**Approach**:
- **Training data**: 30,000 FEA simulations (geometry → field distribution)
- **Architecture**: Encoder-decoder CNN with dilated convolutions
- **Loss function**: $\mathcal{L} = \text{MSE}(B_{\text{pred}}, B_{\text{FEA}})$
- **Physics**: Implicitly learned from training data patterns

**Strengths** {cite}`lecun2015deep`:
- **High accuracy**: 0.3-0.6% NMSE when sufficient data available
- **Fast inference**: 20-30 ms per geometry (30,000× faster than FEA)
- **Complex phenomena**: Handles nonlinearity, saturation, multi-material naturally
- **Mature tooling**: Well-established training pipelines, debugging tools

**Limitations** {cite}`bishop2006pattern,goodfellow2016deep`:
- **Data hungry**: Requires 10,000-50,000 labeled samples (expensive FEA cost)
- **No physics guarantee**: May violate ∇·**B** = 0 locally (divergence errors up to 5%)
- **Poor extrapolation**: Epistemic uncertainty high outside training distribution
- **Black box**: Difficult to interpret why predictions are made

### Physics-Informed Neural Networks (Section 6: PINNs)

**Approach** {cite}`raissi2019physics,karniadakis2021physics`:
- **Training data**: Sparse observations (50-100 points) + Maxwell's equations
- **Architecture**: Feedforward network with automatic differentiation
- **Loss function**: $\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda \mathcal{L}_{\text{physics}}$
- **Physics**: Explicitly enforced through PDE residuals

**Strengths** {cite}`raissi2019physics,cai2021physics`:
- **Data efficient**: 100-500× less labeled data required
- **Physics consistent**: Predictions satisfy Maxwell's equations by construction (residual < 10⁻³)
- **Better extrapolation**: Physical constraints provide inductive bias beyond data
- **Inverse problems**: Can identify material properties, source currents from field measurements
- **Interpretable**: Physics loss components reveal where/why predictions fail

**Limitations** {cite}`wang2021understanding,karniadakis2021physics`:
- **Training complexity**: Multi-objective optimization (data + physics)
- **Computational cost**: Automatic differentiation adds 5-10× training overhead
- **Hyperparameter sensitivity**: Physics weight $\lambda$ requires careful tuning
- **May sacrifice accuracy**: Physics constraints can prevent overfitting to noisy data

:::{important}
**Quantitative Comparison on IPM Motor Field Prediction**  

| Metric | Traditional CNN | PINN |
|--------|----------------|------|
| **Training samples** | 30,000 FEA runs | 100 sparse measurements |
| **Training time** | 15 hours (GPU) | 40 hours (GPU, autodiff overhead) |
| **Inference time** | 25 ms | 30 ms |
| **Accuracy (NMSE)** | 0.6% | 1.2% (trades accuracy for consistency) |
| **Physics consistency** | ∇·**B** error: 3-5% | ∇·**B** error: < 0.1% ✓ |
| **Extrapolation (J = 2× training)** | 15% error | 5% error ✓ |
| **Data generation cost** | $150k (30k FEA × $5) | $500 (100 FEA × $5) |

**Key insight**: PINNs trade 2× accuracy for 300× data efficiency and guaranteed physics consistency.
:::

### When to Use Each Approach?

**Use Traditional DL (CNN) when** {cite}`goodfellow2016deep,lecun2015deep`:
- ✓ Large labeled dataset available (or can afford to generate)
- ✓ Highest accuracy required (< 0.5% error needed)
- ✓ Complex nonlinear phenomena difficult to express analytically
- ✓ Interpolation within well-sampled design space
- ✓ Production deployment requires maximum speed

**Use PINNs when** {cite}`raissi2019physics,karniadakis2021physics`:
- ✓ Limited data available (expensive experiments/simulations)
- ✓ Physics consistency is critical (regulatory, safety-critical)
- ✓ Extrapolation to new parameter ranges required
- ✓ Solving inverse problems (parameter identification)
- ✓ Exploring novel geometries outside training distribution
- ✓ Interpretability matters (need to understand failure modes)

**Hybrid Approach** {cite}`yang2021adversarial,geneva2020modeling` (Recommended):

Combine both paradigms for maximum benefit:
1. **Phase 1 - PINN pretraining**: Learn physics-consistent representations (1-2 weeks)
2. **Phase 2 - Data finetuning**: Refine with limited FEA data (1 week)
3. **Result**: Physics consistency (∇·**B** < 0.1%) + high accuracy (< 0.5% NMSE)

**Implementation strategy**:
```python
# Pseudocode for hybrid training
model = PINN()

# Phase 1: Physics pretraining (sparse collocation)
for epoch in range(5000):
    loss = lambda_physics * physics_loss(model) + data_loss(model, sparse_data)

# Phase 2: Data finetuning (freeze physics-informed layers)
for epoch in range(1000):
    loss = data_loss(model, dense_FEA_data)
```

**Benefit**: Best of both worlds—physics constraints prevent unphysical predictions while data ensures accuracy where it matters {cite}`yang2021adversarial`.

:::{seealso}
**Related Work on Hybrid Physics-Data Approaches**  
- **DeepONet** {cite}`lu2021deeponet`: Operator learning with physics
- **Fourier Neural Operator** {cite}`li2020fourier`: Resolution-invariant physics learning
- **Multi-fidelity modeling** {cite}`meng2020multi,fernandez2016review`: Combine analytical (cheap) + numerical (accurate) data
:::

## Scope of PINN Section (Sections 6a-6d)

This section provides a complete, hands-on introduction to Physics-Informed Neural Networks for electromagnetic field prediction, structured across four subsections.

### Section 6a: Introduction & Literature (Current)

**Content covered**:
- PINN fundamentals and mathematical framework {cite}`raissi2019physics,karniadakis2021physics`
- Comprehensive literature review of PINNs in electromagnetics {cite}`barmada2020deep,gao2021physics,chen2022pinn`
- Comparison with traditional ML (Sections 4-5) and FEA {cite}`ida2015numerical,meeker2015femm`
- When to use PINNs vs. data-driven approaches

### Section 6b: Methodology & Problem Formulation

**Content to be covered**:
- **Maxwell's equations in 2D magnetostatics** {cite}`sadiku2014elements`:
  - Governing PDEs: $\nabla \times \mathbf{H} = \mathbf{J}$, $\nabla \cdot \mathbf{B} = 0$
  - Boundary conditions: Dirichlet, Neumann, interface conditions
- **Canonical problem**: Magnetic field around current-carrying wire
- **Analytical solution** {cite}`ida2015numerical`: $B(r) = \frac{\mu_0 I}{2\pi r}$ (validation reference)
- **Domain specification**: Computational domain, material properties, source terms
- **Collocation strategy**: Sampling interior points and boundary points

### Section 6c: Implementation & Training

**Content to be covered**:
- **PINN architecture**: Feedforward network with automatic differentiation {cite}`raissi2019physics,baydin2018automatic`
- **Physics loss computation**:
  - PDE residual: $\mathcal{L}_{\text{PDE}} = ||\\nabla \\times f_{\\theta}(x,y) - J(x,y)||^2$
  - Boundary loss: $\mathcal{L}_{\text{BC}} = ||f_{\\theta}(x_{\\text{boundary}}) - B_{\\text{boundary}}||^2$
  - Total loss: $\mathcal{L} = \mathcal{L}_{\\text{data}} + \lambda_1 \mathcal{L}_{\\text{PDE}} + \lambda_2 \mathcal{L}_{\\text{BC}}$
- **Training pipeline**:
  - Collocation point generation (Latin Hypercube Sampling)
  - Optimizer selection (Adam vs. L-BFGS) {cite}`kingma2014adam,liu1989limited`
  - Loss balancing strategies {cite}`wang2021understanding`
- **Code pedagogy**: Complete PyTorch implementation with detailed comments
- **Training monitoring**: Loss decomposition, residual visualization

### Section 6d: Results & Discussion

**Content to be covered**:
- **Field distribution predictions**: Comparison with analytical solution
- **Vector field visualizations**: Quiver plots showing field lines
- **Quantitative error analysis**:
  - Point-wise error: $|B_{\\text{PINN}} - B_{\\text{analytical}}|$
  - Relative error: $\\frac{||B_{\\text{PINN}} - B_{\\text{analytical}}||}{||B_{\\text{analytical}}||}$
  - Physics consistency: $|\\nabla \\cdot \\mathbf{B}|$ (should be ≈ 0)
- **Comparison with analytical solutions**: Validate PINN predictions
- **Integration with Chapter 2 architecture**: How PINN fits with CNN approach
- **Discussion**: Lessons learned, limitations, future directions

### Learning Objectives

By completing Sections 6a-6d, you will be able to:

**Theoretical understanding**:
1. Explain how PINNs incorporate physics constraints into neural network training {cite}`raissi2019physics`
2. Formulate electromagnetic problems as physics-informed learning tasks
3. Understand trade-offs between data-driven and physics-informed approaches

**Practical skills**:
4. Implement PINN loss functions with automatic differentiation {cite}`baydin2018automatic`
5. Balance data loss and physics loss for stable training {cite}`wang2021understanding`
6. Validate PINN predictions against analytical/numerical benchmarks
7. Interpret physics loss components to diagnose training failures

**Engineering judgment**:
8. Decide when PINNs are appropriate vs. traditional ML or FEA {cite}`karniadakis2021physics`
9. Design hybrid workflows combining PINNs with data-driven methods {cite}`yang2021adversarial`
10. Apply uncertainty quantification (Section 5) to PINN predictions {cite}`yang2021adversarial,psaros2023uncertainty`

:::{tip}
**Pedagogical Approach**  

This PINN section follows a **learn-by-doing** philosophy:
1. **Conceptual motivation** (6a): Why PINNs? When are they useful?
2. **Mathematical rigor** (6b): Formal problem statement with PDEs
3. **Hands-on implementation** (6c): Complete working code with explanations
4. **Critical evaluation** (6d): Results, validation, lessons learned

**Recommendation**: Work through sections sequentially, running code as you read. Understanding emerges from experimentation {cite}`goodfellow2016deep,karniadakis2021physics`.
:::

### Connection to Broader Thesis Narrative

**Chapter 2 progression**:
- **Sections 1-2**: Motivation and modeling fundamentals
- **Sections 3-5**: Data-driven CNN approach (pure ML, high data requirement)
- **Section 6 (PINNs)**: Physics-informed alternative (low data, explicit physics)
- **Chapter 3**: Real-world applications and case studies

**Key insight**: PINNs and data-driven CNNs are **complementary**, not competing—optimal workflows combine both {cite}`yang2021adversarial,geneva2020modeling`.

Let's proceed to Section 6b to formulate the electromagnetic problem mathematically.